# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [1]:
!pip install -r requirements.txt

In [2]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import trackio
from optuna.visualization.matplotlib import plot_contour, plot_optimization_history
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / 'models'
PLOTS_DIR = PROJECT_ROOT / 'plots'
SCREENSHOTS_DIR = PROJECT_ROOT / 'screenshots'
TRACKIO_PROJECT = 'urban-nest-rent-prediction'
TARGET_COLUMN = 'price'
RANDOM_STATE = 42
CV_FOLDS = 5

for directory in [MODELS_DIR, PLOTS_DIR, SCREENSHOTS_DIR]:
    directory.mkdir(exist_ok=True)

GRID_SEARCH_SPACE = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 8],
}

RANDOM_BAYES_SPACE = {
    'n_estimators': (50, 200),
    'max_depth': (10, 30),
    'min_samples_split': (2, 10),
}

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [3]:
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

feature_columns = [column for column in train_df.columns if column != TARGET_COLUMN]
categorical_columns = [column for column in feature_columns if train_df[column].dtype == 'object']
numerical_columns = [column for column in feature_columns if column not in categorical_columns]

label_encoders = {}
for column in categorical_columns:
    encoder = LabelEncoder()
    combined_values = pd.concat([train_df[column].astype(str), test_df[column].astype(str)], ignore_index=True)
    encoder.fit(combined_values)
    label_encoders[column] = encoder
    train_df[column] = encoder.transform(train_df[column].astype(str))
    test_df[column] = encoder.transform(test_df[column].astype(str))

X_train = train_df[feature_columns]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[feature_columns]
y_test = test_df[TARGET_COLUMN]

ui_metadata = {
    'feature_columns': feature_columns,
    'categorical_columns': categorical_columns,
    'numerical_columns': numerical_columns,
    'target_column': TARGET_COLUMN,
    'category_options': {column: label_encoders[column].classes_.tolist() for column in categorical_columns},
    'numeric_features': {},
}

raw_train_df = pd.read_csv('Dataset/train.csv')
for column in numerical_columns:
    series = raw_train_df[column]
    is_integer = pd.api.types.is_integer_dtype(series)
    ui_metadata['numeric_features'][column] = {
        'min': int(series.min()) if is_integer else float(series.min()),
        'max': int(series.max()) if is_integer else float(series.max()),
        'default': int(series.median()) if is_integer else float(series.median()),
        'dtype': 'int' if is_integer else 'float',
    }

print('Feature columns:', feature_columns)
print('Categorical columns:', categorical_columns)
print('Numerical columns:', numerical_columns)

Feature columns: ['location', 'city', 'latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Status', 'Size_ft²', 'BHK', 'rooms_num', 'property_type', 'verification_days']
Categorical columns: ['location', 'city', 'Status', 'property_type']
Numerical columns: ['latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Size_ft²', 'BHK', 'rooms_num', 'verification_days']


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [4]:
rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

def build_history_from_cv_results(method_name, cv_results):
    errors = -np.array(cv_results['mean_test_score'])
    best_errors = np.minimum.accumulate(errors)
    history = []
    for index, params in enumerate(cv_results['params'], start=1):
        history.append({
            'iteration': index,
            'cv_mae': float(errors[index - 1]),
            'best_cv_mae': float(best_errors[index - 1]),
            'n_estimators': int(params['n_estimators']),
            'max_depth': int(params['max_depth']),
            'min_samples_split': int(params['min_samples_split']),
            'method': method_name,
        })
    return history

def log_history_to_trackio(method_name, history, best_params, best_cv_mae, elapsed_seconds):
    trackio.init(
        project=TRACKIO_PROJECT,
        name=method_name.lower().replace(' ', '_'),
        config={
            'method': method_name,
            'cv_folds': CV_FOLDS,
            'total_trials': len(history),
        },
    )
    for row in history:
        trackio.log(row)
    trackio.log({
        'summary/best_cv_mae': float(best_cv_mae),
        'summary/elapsed_seconds': float(elapsed_seconds),
        'summary/best_n_estimators': int(best_params['n_estimators']),
        'summary/best_max_depth': int(best_params['max_depth']),
        'summary/best_min_samples_split': int(best_params['min_samples_split']),
    })
    trackio.finish()

# 1. Grid Search Implementation
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=GRID_SEARCH_SPACE,
    scoring='neg_mean_absolute_error',
    cv=CV_FOLDS,
    n_jobs=-1,
    return_train_score=False,
)
grid_start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_elapsed = time.perf_counter() - grid_start
grid_history = build_history_from_cv_results('Grid Search', grid_search.cv_results_)
grid_best_params = {key: int(value) for key, value in grid_search.best_params_.items()}
grid_best_cv_mae = float(-grid_search.best_score_)
log_history_to_trackio('Grid Search', grid_history, grid_best_params, grid_best_cv_mae, grid_elapsed)

# 2. Random Search Implementation
random_distributions = {
    'n_estimators': list(range(RANDOM_BAYES_SPACE['n_estimators'][0], RANDOM_BAYES_SPACE['n_estimators'][1] + 1)),
    'max_depth': list(range(RANDOM_BAYES_SPACE['max_depth'][0], RANDOM_BAYES_SPACE['max_depth'][1] + 1)),
    'min_samples_split': list(range(RANDOM_BAYES_SPACE['min_samples_split'][0], RANDOM_BAYES_SPACE['min_samples_split'][1] + 1)),
}
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_distributions,
    n_iter=60,
    scoring='neg_mean_absolute_error',
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=False,
)
random_start = time.perf_counter()
random_search.fit(X_train, y_train)
random_elapsed = time.perf_counter() - random_start
random_history = build_history_from_cv_results('Random Search', random_search.cv_results_)
random_best_params = {key: int(value) for key, value in random_search.best_params_.items()}
random_best_cv_mae = float(-random_search.best_score_)
log_history_to_trackio('Random Search', random_history, random_best_params, random_best_cv_mae, random_elapsed)

# 3. Bayesian Optimization (Optuna) Implementation
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', *RANDOM_BAYES_SPACE['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *RANDOM_BAYES_SPACE['max_depth']),
        'min_samples_split': trial.suggest_int('min_samples_split', *RANDOM_BAYES_SPACE['min_samples_split']),
    }
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_absolute_error', cv=CV_FOLDS, n_jobs=1)
    return float(-scores.mean())

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
bayes_start = time.perf_counter()
study.optimize(objective, n_trials=60, show_progress_bar=False)
bayes_elapsed = time.perf_counter() - bayes_start

bayes_history = []
best_so_far = float('inf')
for index, trial in enumerate(study.trials, start=1):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        continue
    best_so_far = min(best_so_far, float(trial.value))
    bayes_history.append({
        'iteration': index,
        'cv_mae': float(trial.value),
        'best_cv_mae': float(best_so_far),
        'n_estimators': int(trial.params['n_estimators']),
        'max_depth': int(trial.params['max_depth']),
        'min_samples_split': int(trial.params['min_samples_split']),
        'method': 'Bayesian Optimization',
    })
bayes_best_params = {key: int(value) for key, value in study.best_params.items()}
bayes_best_cv_mae = float(study.best_value)
log_history_to_trackio('Bayesian Optimization', bayes_history, bayes_best_params, bayes_best_cv_mae, bayes_elapsed)

results = [
    {
        'method': 'Grid Search',
        'history': grid_history,
        'best_params': grid_best_params,
        'best_cv_mae': grid_best_cv_mae,
        'elapsed_seconds': grid_elapsed,
    },
    {
        'method': 'Random Search',
        'history': random_history,
        'best_params': random_best_params,
        'best_cv_mae': random_best_cv_mae,
        'elapsed_seconds': random_elapsed,
    },
    {
        'method': 'Bayesian Optimization',
        'history': bayes_history,
        'best_params': bayes_best_params,
        'best_cv_mae': bayes_best_cv_mae,
        'elapsed_seconds': bayes_elapsed,
    },
]

pd.DataFrame([
    {
        'method': result['method'],
        'best_cv_mae': result['best_cv_mae'],
        'elapsed_seconds': result['elapsed_seconds'],
        'best_params': result['best_params'],
    }
    for result in results
])

* Trackio project initialized: urban-nest-rent-prediction
* Trackio metrics logged to: C:\Users\RAJDEEP VRAJ\.cache\huggingface\trackio
* Created new run: dainty-sunset-0


C:\Users\RAJDEEP VRAJ\AppData\Local\Programs\Python\Python311\Lib\site-packages\trackio\__init__.py:287: UserWarning: * Warning: resume='never' but a run 'grid_search' already exists in project 'urban-nest-rent-prediction'. Generating a new name and instead. If you want to resume this run, call init() with resume='must' or resume='allow'.
  warnings.warn(


* Run finished. Uploading logs to Trackio (please wait...)


C:\Users\RAJDEEP VRAJ\AppData\Local\Programs\Python\Python311\Lib\site-packages\trackio\__init__.py:287: UserWarning: * Warning: resume='never' but a run 'random_search' already exists in project 'urban-nest-rent-prediction'. Generating a new name and instead. If you want to resume this run, call init() with resume='must' or resume='allow'.
  warnings.warn(


* Created new run: brave-forest-1
* Run finished. Uploading logs to Trackio (please wait...)


[I 2026-04-15 00:59:37,272] A new study created in memory with name: no-name-27e2af5e-1f53-4f86-bbc9-96d0864c785b
[I 2026-04-15 00:59:53,992] Trial 0 finished with value: 13853.402254485176 and parameters: {'n_estimators': 106, 'max_depth': 29, 'min_samples_split': 8}. Best is trial 0 with value: 13853.402254485176.
[I 2026-04-15 01:00:09,126] Trial 1 finished with value: 14096.243211520685 and parameters: {'n_estimators': 140, 'max_depth': 13, 'min_samples_split': 3}. Best is trial 0 with value: 13853.402254485176.
[I 2026-04-15 01:00:16,885] Trial 2 finished with value: 13754.029656093433 and parameters: {'n_estimators': 58, 'max_depth': 28, 'min_samples_split': 7}. Best is trial 2 with value: 13754.029656093433.
[I 2026-04-15 01:00:29,499] Trial 3 finished with value: 15488.680105230416 and parameters: {'n_estimators': 156, 'max_depth': 10, 'min_samples_split': 10}. Best is trial 2 with value: 13754.029656093433.
[I 2026-04-15 01:00:44,050] Trial 4 finished with value: 13858.9956024

* Created new run: calm-river-2
* Run finished. Uploading logs to Trackio (please wait...)


C:\Users\RAJDEEP VRAJ\AppData\Local\Programs\Python\Python311\Lib\site-packages\trackio\__init__.py:287: UserWarning: * Warning: resume='never' but a run 'bayesian_optimization' already exists in project 'urban-nest-rent-prediction'. Generating a new name and instead. If you want to resume this run, call init() with resume='must' or resume='allow'.
  warnings.warn(


,method,best_cv_mae,elapsed_seconds,best_params
0,Grid Search,13270.758152,334.511400,"{'max_depth': 25, 'min_samples_split': 2, 'n_e..."
1,Random Search,13299.757428,327.467915,"{'n_estimators': 142, 'min_samples_split': 2, ..."
2,Bayesian Optimization,13268.449001,827.669441,"{'n_estimators': 184, 'max_depth': 26, 'min_sa..."


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [5]:
figure, axis = plt.subplots(figsize=(12.5, 6.5))
for result in results:
    history_df = pd.DataFrame(result['history'])
    axis.plot(history_df['iteration'], history_df['best_cv_mae'], linewidth=2.2, label=result['method'])
axis.set_xlabel('Number of iterations / trials')
axis.set_ylabel('Best mean CV MAE so far')
axis.set_title('Optimization Budget vs. Best Cross-Validation Error')
axis.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, frameon=True)
figure.subplots_adjust(right=0.78)
trials_plot_path = PLOTS_DIR / 'trials_vs_error.png'
figure.savefig(trials_plot_path, dpi=200, bbox_inches='tight')
plt.show()
plt.close(figure)

try:
    contour_axes = plot_contour(study, params=['n_estimators', 'max_depth', 'min_samples_split'])
    optuna_figure = contour_axes.flatten()[0].figure if hasattr(contour_axes, 'flatten') else contour_axes.figure
except Exception:
    fallback_axis = plot_optimization_history(study)
    optuna_figure = fallback_axis.figure
optuna_figure.set_size_inches(11, 8)
optuna_figure.subplots_adjust(left=0.08, right=0.95, top=0.92, bottom=0.08, wspace=0.22, hspace=0.22)
optuna_plot_path = PLOTS_DIR / 'optuna_hyperparameter_space.png'
optuna_figure.savefig(optuna_plot_path, dpi=200, bbox_inches='tight')
plt.show()
plt.close(optuna_figure)

print('Saved:', trials_plot_path)
print('Saved:', optuna_plot_path)

C:\Users\RAJDEEP VRAJ\AppData\Local\Temp\ipykernel_26520\2774711427.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\RAJDEEP VRAJ\AppData\Local\Temp\ipykernel_26520\2774711427.py:16: ExperimentalWarning: optuna.visualization.matplotlib._contour.plot_contour is experimental (supported from v2.2.0). The interface can change in the future.
  contour_axes = plot_contour(study, params=['n_estimators', 'max_depth', 'min_samples_split'])


Saved: C:\Users\RAJDEEP VRAJ\Documents\New project\STTAI2026-Assignment4-main\plots\trials_vs_error.png
Saved: C:\Users\RAJDEEP VRAJ\Documents\New project\STTAI2026-Assignment4-main\plots\optuna_hyperparameter_space.png


C:\Users\RAJDEEP VRAJ\AppData\Local\Temp\ipykernel_26520\2774711427.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Final Testing & Model Saving
Report the best hyperparameters found by all 3 methods, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [6]:
for result in results:
    print(f"{result['method']}: {result['best_params']} | Best CV MAE = {result['best_cv_mae']:.4f}")

best_overall = min(results, key=lambda result: result['best_cv_mae'])
best_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_overall['best_params'])
best_model.fit(X_train, y_train)

test_predictions = best_model.predict(X_test)
final_test_mae = mean_absolute_error(y_test, test_predictions)
print(f"\nOverall best method: {best_overall['method']}")
print(f"Final Test MAE: {final_test_mae:.4f}")

with open(MODELS_DIR / 'best_rf_model.pkl', 'wb') as model_file:
    pickle.dump(best_model, model_file)

encoder_payload = {
    'label_encoders': label_encoders,
    'ui_metadata': ui_metadata,
    'overall_best_method': best_overall['method'],
    'test_mae': float(final_test_mae),
    'best_results': {
        result['method']: {
            'best_params': result['best_params'],
            'best_cv_mae': result['best_cv_mae'],
            'elapsed_seconds': result['elapsed_seconds'],
        }
        for result in results
    },
}
with open(MODELS_DIR / 'label_encoders.pkl', 'wb') as encoder_file:
    pickle.dump(encoder_payload, encoder_file)

summary_payload = {
    'trackio_project': TRACKIO_PROJECT,
    'overall_best_method': best_overall['method'],
    'best_hyperparameters': best_overall['best_params'],
    'test_mae': float(final_test_mae),
    'methods': {
        result['method']: {
            'best_params': result['best_params'],
            'best_cv_mae': result['best_cv_mae'],
            'elapsed_seconds': result['elapsed_seconds'],
        }
        for result in results
    },
}
with open(MODELS_DIR / 'experiment_summary.json', 'w', encoding='utf-8') as summary_file:
    json.dump(summary_payload, summary_file, indent=2)

print('Saved model to:', MODELS_DIR / 'best_rf_model.pkl')
print('Saved encoders to:', MODELS_DIR / 'label_encoders.pkl')
print('Saved summary to:', MODELS_DIR / 'experiment_summary.json')
print("\nOpen the Trackio dashboard and save a screenshot as screenshots/trackio_dashboard.png")
print(f"Command: trackio show --project {TRACKIO_PROJECT}")

Grid Search: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200} | Best CV MAE = 13270.7582
Random Search: {'n_estimators': 142, 'min_samples_split': 2, 'max_depth': 24} | Best CV MAE = 13299.7574
Bayesian Optimization: {'n_estimators': 184, 'max_depth': 26, 'min_samples_split': 2} | Best CV MAE = 13268.4490

Overall best method: Bayesian Optimization
Final Test MAE: 12418.9658
Saved model to: C:\Users\RAJDEEP VRAJ\Documents\New project\STTAI2026-Assignment4-main\models\best_rf_model.pkl
Saved encoders to: C:\Users\RAJDEEP VRAJ\Documents\New project\STTAI2026-Assignment4-main\models\label_encoders.pkl
Saved summary to: C:\Users\RAJDEEP VRAJ\Documents\New project\STTAI2026-Assignment4-main\models\experiment_summary.json

Open the Trackio dashboard and save a screenshot as screenshots/trackio_dashboard.png
Command: trackio show --project urban-nest-rent-prediction
